In [1]:
import os, sys

WORK  = '/kaggle/working'
REPO  = 'https://github.com/PhuongThao-2005/TULIP-MedML.git'
RNAME = 'TULIP-MedML'
BRANCH = 'test_c1'

if os.path.exists(f'{WORK}/{RNAME}'):
    os.system(f'cd {WORK}/{RNAME} && git fetch origin')
    os.system(f'cd {WORK}/{RNAME} && git checkout {BRANCH}')
    os.system(f'cd {WORK}/{RNAME} && git pull origin {BRANCH}')
    print(f'Pulled latest ({BRANCH})')
else:
    os.system(f'cd {WORK} && git clone -b {BRANCH} {REPO}')
    print(f'Cloned ({BRANCH})')

os.system('pip install pyyaml timm -q')
sys.path.insert(0, f'{WORK}/{RNAME}')
os.chdir(f'{WORK}/{RNAME}')

print('Working dir :', os.getcwd())
print('Repo contents:', os.listdir('.'))

Cloning into 'TULIP-MedML'...


Cloned (test_c1)
Working dir : /kaggle/working/TULIP-MedML
Repo contents: ['requirements.txt', '.git', 'notebooks', 'src', '.gitignore', 'data', 'tulip-medml-train.ipynb', 'README.md']


In [2]:
# import shutil
# import os

# SRC = "/kaggle/input/tulip-medml-train/checkpoints/c1"
# DST = "/kaggle/working/checkpoints/c1"

# if os.path.exists(SRC):
#     shutil.copytree(SRC, DST, dirs_exist_ok=True)
#     print(" Copied checkpoints to working")
# else:
#     print("None")

In [3]:
# ── Cell 2: Kiểm tra GPU + dataset ───────────────────────────────────
import torch, pandas as pd
 
print('GPU  :', torch.cuda.get_device_name(0))
print('VRAM :', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')
 
ROOT = '/kaggle/input/datasets/ashery/chexpert'
df   = pd.read_csv(f'{ROOT}/train.csv')
 
# Path trong CSV có dạng CheXpert-v1.0-small/... → bỏ phần đầu
sample = '/'.join(df['Path'].iloc[0].split('/')[1:])
abs_p  = os.path.join(ROOT, sample)
 
print('\nDataset root :', ROOT)
print('train.csv    :', len(df), 'rows')
print('val.csv      :', len(pd.read_csv(f'{ROOT}/valid.csv')), 'rows')
print('Image exists :', os.path.isfile(abs_p))
 
print('\nData files in repo:')
for fname in ['data/chexpert_glove_word2vec.npy', 'data/chexpert_adj.pkl']:
    exists = os.path.isfile(fname)
    size   = os.path.getsize(fname) / 1024 if exists else 0
    print(f'  {fname:45s} {"OK" if exists else "MISSING":8s} {size:.1f} KB')


GPU  : Tesla T4
VRAM : 15.6 GB

Dataset root : /kaggle/input/datasets/ashery/chexpert
train.csv    : 223414 rows
val.csv      : 234 rows
Image exists : True

Data files in repo:
  data/chexpert_glove_word2vec.npy              OK       16.5 KB
  data/chexpert_adj.pkl                         OK       1.0 KB


In [4]:
# ── Cell 3: Build adjacency matrix (nếu chưa có) ──────────────────────────────
# Co-occurrence được tính chỉ trên nhãn = 1 (positive).
# Uncertain (-1) và negative (0) đều không đếm là co-occur,
# để tránh inflate co-occurrence của các bệnh hiếm / không rõ.
import numpy as np, pickle
from src.data.chexpert import CHEXPERT_CLASSES
 
ADJ_PATH = 'data/chexpert_adj.pkl'
os.makedirs('data', exist_ok=True)
 
if not os.path.isfile(ADJ_PATH):
    print('Building adjacency matrix from train.csv ...')
    df_adj = pd.read_csv(f'{ROOT}/train.csv')
    labels = df_adj[CHEXPERT_CLASSES].fillna(0).values  # NaN → 0
 
    # Chỉ tính co-occur khi cả hai label == 1; uncertain (-1) → 0
    labels_bin = (labels == 1).astype(np.float32)       # {0, 1}
 
    n = len(CHEXPERT_CLASSES)
    adj  = np.zeros((n, n), dtype=np.float32)
    nums = np.zeros(n,      dtype=np.float32)
 
    for row in labels_bin:
        pos = np.where(row == 1)[0]
        nums[pos] += 1
        for i in pos:
            for j in pos:
                adj[i, j] += 1
 
    pickle.dump({'adj': adj, 'nums': nums}, open(ADJ_PATH, 'wb'))
    print(f'  Saved → {ADJ_PATH}  (shape {adj.shape})')
else:
    print(f'adj exists: {ADJ_PATH}')

adj exists: data/chexpert_adj.pkl


In [ ]:
!cd /kaggle/working/TULIP-MedML && python src/train.py \
    --config src/configs/c1.yaml

Config: C1_baseline
[CheXpert] 22341 samples | policy=keep
[CheXpert] 234 samples | policy=keep
[CheXpert] 500 samples | policy=keep
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet101_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet101_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Downloading: "https://download.pytorch.org/models/resnet101-63fe2227.pth" to /root/.cache/torch/hub/checkpoints/resnet101-63fe2227.pth
100%|█████████████████████████████████████████| 171M/171M [00:00<00:00, 229MB/s]
Train from scratch
/kaggle